In [4]:
import pandas as pd
import numpy as np
import re
from collections import Counter

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text.split()


df = pd.read_csv('Dataset/Derby_Original_train.csv')
df['full_text'] = df['summary'].fillna('') + " " + df['description'].fillna('')


all_tokens = []
for text in df['full_text']:
    all_tokens.extend(preprocess_text(text))


word_counts = Counter(all_tokens)
vocab = [word for word, count in word_counts.items() if count > 5]
word2idx = {word: i for i, word in enumerate(vocab)}
idx2word = {i: word for word, i in word2idx.items()}
vocab_size = len(vocab)
data_ids = [word2idx[w] for w in all_tokens if w in word2idx]

In [ ]:
def generate_batch(data, window_size, num_neg_samples):
    for i, target_idx in enumerate(data):
        start = max(0, i - window_size)
        end = min(len(data), i + window_size + 1)
        contexts = [data[j] for j in range(start, end) if i != j]
        
        for context_idx in contexts:
            neg_samples = np.random.randint(0, vocab_size, num_neg_samples)
            yield target_idx, context_idx, neg_samples

embedding_dim = 100
learning_rate = 0.01
window_size = 2
num_neg_samples = 5

In [ ]:
W_in = np.random.uniform(-0.5, 0.5, (vocab_size, embedding_dim)) / embedding_dim
W_out = np.random.uniform(-0.5, 0.5, (vocab_size, embedding_dim)) / embedding_dim

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -10, 10)))

In [ ]:
epochs = 5
for epoch in range(epochs):
    total_loss = 0
    
    for target, context, negatives in generate_batch(data_ids, window_size, num_neg_samples):
        v_target = W_in[target]
        pos_score = sigmoid(np.dot(v_target, W_out[context]))
        neg_vectors = W_out[negatives]
        neg_scores = sigmoid(-np.dot(neg_vectors, v_target))
        loss = -np.log(pos_score + 1e-9) - np.sum(np.log(neg_scores + 1e-9))
        total_loss += loss
        grad_out_pos = (pos_score - 1) * v_target
        grad_out_neg = (1 - neg_scores).reshape(-1, 1) * v_target
        grad_in = (pos_score - 1) * W_out[context] + \
                  np.dot((1 - neg_scores), neg_vectors)
        
        W_in[target] -= learning_rate * grad_in
        W_out[context] -= learning_rate * grad_out_pos
        W_out[negatives] -= learning_rate * grad_out_neg
        
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(data_ids):.4f}")

Epoch 1/5, Loss: 13.3128
Epoch 2/5, Loss: 8.1229
Epoch 3/5, Loss: 7.3702
Epoch 4/5, Loss: 7.1149
Epoch 5/5, Loss: 6.9540


In [9]:
def get_similar_words(query_word, top_n=10):
    if query_word not in word2idx:
        return f"Word '{query_word}' not in vocabulary."

    
    query_idx = word2idx[query_word]
    query_vec = W_in[query_idx]

    norms = np.linalg.norm(W_in, axis=1, keepdims=True)
    norm_W = W_in / (norms + 1e-9)
    query_vec_norm = query_vec / (np.linalg.norm(query_vec) + 1e-9)
    similarities = np.dot(norm_W, query_vec_norm)


    nearest_indices = similarities.argsort()[::-1][1:top_n+1]

    print(f"Words most similar to '{query_word}' are: ")
    for idx in nearest_indices:
        print(f"  - {idx2word[idx]}: {similarities[idx]:.4f}")

get_similar_words("exception")

Words most similar to 'exception' are: 
  - end: 0.9713
  - java: 0.9659
  - parameter: 0.9591
  - javalangnullpointerexception: 0.9464
  - main: 0.9426
  - gmt: 0.9400
  - orgapachederbysharedcommonsanitysanitymanagerthrowassertsanitymanagerjava: 0.9394
  - orgapachederbyimpljdbcsqlexceptionfactorygetsqlexceptionsqlexceptionfactoryjava: 0.9385
  - engine: 0.9382
  - access: 0.9378
